# Evaluation: query Types 1–6

Loads per-type qrels and queries, evaluates **RRF** (primary) and **BM25** (mandatory baseline), optional **dense** run, and exports CSV summaries under `results/evaluation/`.

Metric policy is defined in `src/evaluation/utils.py` (`TYPE_METRICS`, `OVERALL_METRICS`).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(obj):
        print(obj)


# Project root (works if cwd is repo root or notebooks/)
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "qrels").exists() and (ROOT.parent / "data" / "qrels").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.utils import (
    OVERALL_METRICS,
    TYPE_METRICS,
    build_bm25_rrf_comparison_table,
    default_data_paths,
    evaluate_all_types,
    evaluate_overall_types_1_6,
    evaluate_per_query_all_types,
    load_run_csv,
    merge_all_types_qrels,
    merge_all_types_queries,
    round_metrics_df,
    validate_run_coverage,
)

print("PROJECT_ROOT =", ROOT)
print("TYPE_METRICS keys:", sorted(TYPE_METRICS.keys()))

## Configuration

Set paths to your ranked-list CSVs. Expected columns: `query_id`, `doc_id`, `rank` (and for RRF: `rrf_score`; for BM25 often `score` or `bm25_score`).

In [ ]:
# --- Required: final fused run and BM25 baseline ---
RRF_RUN_PATH = ROOT / "results" / "runs" / "rrf_fused.csv"  # adjust
BM25_RUN_PATH = ROOT / "results" / "runs" / "bm25.csv"  # adjust

# Optional: dense retrieval run (same CSV schema)
DENSE_RUN_PATH = ROOT / "results" / "runs" / "dense.csv"  # set to None to skip

QRELS_PATHS, QUERY_PATHS = default_data_paths(ROOT)
# Only Types 1–6
QRELS_PATHS = {t: QRELS_PATHS[t] for t in range(1, 7)}
QUERY_PATHS = {t: QUERY_PATHS[t] for t in range(1, 7)}

OUT_DIR = ROOT / "results" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for label, p in [("RRF", RRF_RUN_PATH), ("BM25", BM25_RUN_PATH)]:
    print(f"{label}: {p}  exists={p.exists()}")
if DENSE_RUN_PATH:
    print(f"Dense: {DENSE_RUN_PATH}  exists={Path(DENSE_RUN_PATH).exists()}")

## Load qrels and queries (Types 1–6)

In [ ]:
qrels = merge_all_types_qrels(QRELS_PATHS)
queries_df = merge_all_types_queries(QUERY_PATHS)

print("Total qrels queries:", len(qrels))
print("Total query rows:", len(queries_df))
print(queries_df["query_type"].value_counts().sort_index())

## Load runs

In [ ]:
run_rrf = load_run_csv(RRF_RUN_PATH, score_col="rrf_score")
run_bm25 = load_run_csv(BM25_RUN_PATH)

run_dense = None
dense_by_type = None
if DENSE_RUN_PATH and Path(DENSE_RUN_PATH).exists():
    run_dense = load_run_csv(DENSE_RUN_PATH)

print("RRF queries in run:", len(run_rrf))
print("BM25 queries in run:", len(run_bm25))

## Validation: coverage vs Types 1–6 query IDs (in qrels)

In [ ]:
mask = queries_df["query_type"].between(1, 6)
eval_qids = [
    q for q in queries_df.loc[mask, "query_id"].astype(str).tolist() if q in qrels
]

cov_rrf = validate_run_coverage(run_rrf, eval_qids, min_depth=10)
cov_bm25 = validate_run_coverage(run_bm25, eval_qids, min_depth=10)
print("RRF missing queries:", cov_rrf["missing"].sum())
print("BM25 missing queries:", cov_bm25["missing"].sum())
display(cov_rrf[cov_rrf["missing"] | cov_rrf["shallow"]].head(20))

## Primary system: RRF — per-type, overall, per-query

In [ ]:
rrf_by_type = evaluate_all_types(qrels, run_rrf, queries_df)
rrf_overall = evaluate_overall_types_1_6(qrels, run_rrf, queries_df)
rrf_per_query = evaluate_per_query_all_types(qrels, run_rrf, queries_df)

display(round_metrics_df(rrf_by_type))
print("RRF overall (Types 1–6):", rrf_overall)

## Baseline: BM25 — per-type, overall, per-query

In [ ]:
bm25_by_type = evaluate_all_types(qrels, run_bm25, queries_df)
bm25_overall = evaluate_overall_types_1_6(qrels, run_bm25, queries_df)
bm25_per_query = evaluate_per_query_all_types(qrels, run_bm25, queries_df)

display(round_metrics_df(bm25_by_type))
print("BM25 overall (Types 1–6):", bm25_overall)

## Report-critical: BM25 vs RRF (deltas per metric)

In [ ]:
compare_by_type = build_bm25_rrf_comparison_table(bm25_by_type, rrf_by_type)
display(round_metrics_df(compare_by_type))

overall_compare = pd.DataFrame(
    [{"system": "BM25", **bm25_overall}, {"system": "RRF", **rrf_overall}]
)
delta = {"system": "RRF_minus_BM25"}
for m in OVERALL_METRICS:
    if m in bm25_overall and m in rrf_overall:
        delta[m] = rrf_overall[m] - bm25_overall[m]
overall_compare = pd.concat(
    [overall_compare, pd.DataFrame([delta])], ignore_index=True
)
display(round_metrics_df(overall_compare))

## Optional: dense run (same evaluation hooks)

In [ ]:
if run_dense:
    dense_by_type = evaluate_all_types(qrels, run_dense, queries_df)
    display(round_metrics_df(dense_by_type))
else:
    print("No dense run loaded — skipped.")
    dense_by_type = None

## Export CSVs for the report

Files go to `results/evaluation/` (gitignored by default).

In [ ]:
round_metrics_df(rrf_by_type).to_csv(OUT_DIR / "types_1_6_rrf_by_type.csv", index=False)
pd.DataFrame([rrf_overall]).to_csv(OUT_DIR / "types_1_6_rrf_overall.csv", index=False)
rrf_per_query.to_csv(OUT_DIR / "types_1_6_rrf_per_query.csv", index=False)

round_metrics_df(bm25_by_type).to_csv(OUT_DIR / "types_1_6_bm25_by_type.csv", index=False)
pd.DataFrame([bm25_overall]).to_csv(OUT_DIR / "types_1_6_bm25_overall.csv", index=False)
bm25_per_query.to_csv(OUT_DIR / "types_1_6_bm25_per_query.csv", index=False)

round_metrics_df(compare_by_type).to_csv(OUT_DIR / "types_1_6_bm25_vs_rrf_by_type.csv", index=False)

# Overall BM25 vs RRF row
ov_comp = pd.DataFrame(
    [
        {"system": "BM25", **bm25_overall},
        {"system": "RRF", **rrf_overall},
    ]
)
if len(ov_comp) == 2:
    delta = {"system": "RRF_minus_BM25"}
    for m in OVERALL_METRICS:
        delta[m] = ov_comp.loc[1, m] - ov_comp.loc[0, m]
    ov_comp = pd.concat([ov_comp, pd.DataFrame([delta])], ignore_index=True)
round_metrics_df(ov_comp).to_csv(OUT_DIR / "types_1_6_bm25_vs_rrf_overall.csv", index=False)

if dense_by_type is not None:
    round_metrics_df(dense_by_type).to_csv(
        OUT_DIR / "types_1_6_dense_by_type.csv", index=False
    )

print("Wrote CSVs to", OUT_DIR)

## Notes

- **Types 3–4** use graded qrels; `ndcg@10` is the primary ranked metric for those types.
- **Types 5–6** emphasize recall at shallow depth.
- Unjudged documents are not in qrels; standard pooling assumption applies for retrieved unjudged docs.
- If a run file is missing, fix `RRF_RUN_PATH` / `BM25_RUN_PATH` above (or generate runs from your retrieval pipeline).